# 98 - DSPy + MCP: prompts y contexto desde SQLite — Base

Simulamos **DSPy** (mejora de prompts) y **MCP** (contexto) con una base SQLite.

Esta celda crea una base SQLite pequeña con documentos de ejemplo. Se usa como fuente local de contexto para simular MCP.


In [ ]:
import sqlite3, os
DB='contexto.db'
if os.path.exists(DB): os.remove(DB)
conn=sqlite3.connect(DB); cur=conn.cursor()
cur.execute('CREATE TABLE docs (id INTEGER PRIMARY KEY, title TEXT, content TEXT)')
cur.executemany('INSERT INTO docs(title,content) VALUES(?,?)', [
    ('Regresión logística','Clasificador lineal con función sigmoide.'),
    ('Redes neuronales','Modelo con capas y neuronas.'),
    ('Gradiente descendente','Optimiza parámetros minimizando pérdida.')
])
conn.commit(); print('✅ BD creada con 3 docs')


Esta celda define dos piezas básicas: una mejora determinista del prompt y una función de recuperación de contexto desde SQLite.


In [ ]:
def improve_prompt(prompt:str)->str:
    p=prompt.strip()
    if not p.endswith('?'): p+='?'
    return 'Responde brevemente y con precisión: '+p

def retrieve_context(term:str, topk=2):
    q=f"SELECT title,content FROM docs WHERE title LIKE ? OR content LIKE ? LIMIT {topk}"
    return cur.execute(q,(f'%{term}%',f'%{term}%')).fetchall()

prompt='explica brevemente redes neuronales'
better=improve_prompt(prompt)
ctx=retrieve_context('neuron')
print('PROMPT:', better); print('CTX:', ctx)


## Ampliación progresiva

MCP puede entenderse como un contrato para entregar contexto estructurado al modelo. Aquí construimos un paquete JSON con consulta, documentos recuperados y metadatos.

Esta celda construye un paquete de contexto con estructura tipo MCP y genera una respuesta simulada basada solo en los documentos recuperados.


In [ ]:
def build_context_packet(query, retriever=retrieve_context, topk=2):
    rows = retriever(query, topk=topk)
    return {
        "query": query,
        "documents": [
            {"title": title, "content": content, "source": "sqlite://contexto.db/docs"}
            for title, content in rows
        ],
        "policy": "Usar solo los documentos proporcionados si son relevantes.",
    }

def grounded_answer(prompt, packet):
    docs = packet["documents"]
    if not docs:
        return "No hay contexto suficiente para responder con seguridad."
    citations = ", ".join(doc["title"] for doc in docs)
    return f"{prompt}\nRespuesta simulada basada en: {citations}."

packet = build_context_packet("neuron")
print(packet)
print(grounded_answer(better, packet))

Esta celda valida si la respuesta generada menciona alguna de las fuentes recuperadas, una comprobación mínima de fundamentación en contexto.


In [ ]:
def validate_grounding(answer, packet):
    titles = [doc["title"] for doc in packet["documents"]]
    used = [title for title in titles if title in answer]
    return {"uses_context": bool(used), "citations_found": used, "num_docs": len(titles)}

answer = grounded_answer(better, packet)
print(validate_grounding(answer, packet))

### Reto adicional

Cambia `retrieve_context` por un ranking TF-IDF y exporta `packet` como JSON para simular una respuesta compatible con MCP.

## Para profundizar
Este notebook introduce DSPy + MCP. La documentación completa incluye:
- Signatures, Modules, Optimizers, RAG context

Consulta el documento correspondiente en Documentacion/.

In [ ]:
# Los ejemplos avanzados están en los documentos de Documentacion/# Consulta el archivo correspondiente según lo que quieras profundizar.